In [3]:
%pip install opencv-python numpy shapely matplotlib scikit-image

  Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 12.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 11.0 MB/s  0:00:00
Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_x86_64.whl (25.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-image] [scikit-image]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import numpy as np
from shapely.geometry import Polygon

from vision import (initialize_camera, initialize_vision, detect_thymio,
                    GridNotDetectedError, camera_id, camera_width, fps,
                    warmup_frames, grid_side_mm)
from global_navigation import (build_free_space, compute_path,
                               draw_obstacles, draw_path, draw_legend)

ARENA_W = ARENA_H = grid_side_mm   # 700 mm, from vision (origin top left corner)


def init_obstacles(frame):
    """Run the vision, inflate the obstacles and show them on the frame.

    Returns H (pixels -> mm), to_camera (mm -> pixels), obstacles, inflated obstacles, goal.
    """
    H, obstacles_mm, goal_mm = initialize_vision(frame)
    to_camera = np.linalg.inv(H)
    obstacles = [Polygon(v) for v in obstacles_mm]
    _, inflated_obstacles, _ = build_free_space(obstacles, ARENA_W, ARENA_H)
    cv2.imshow("Vision: inflated obstacles",
               draw_legend(draw_obstacles(frame.copy(), obstacles, inflated_obstacles, to_camera)))
    return H, to_camera, obstacles, inflated_obstacles, goal_mm


def main():
    cap = initialize_camera(camera_id, camera_width, fps, warmup_frames)
    dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector = cv2.aruco.ArucoDetector(dictionary)
    frame = None

    try:
        # 1. Vision: grid, pieces, best move (once)
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError("Cannot read the camera image")
        H, to_camera, obstacles, inflated_obstacles, goal_mm = init_obstacles(frame)

        waypoints = samples = None
        while True:
            ret, frame = cap.read()
            if not ret:
                raise RuntimeError("Cannot read the camera image")

            # 2. Thymio pose (every frame)
            thymio = detect_thymio(frame, H, detector)

            # 3. Global navigation: once, as soon as the Thymio is seen
            if waypoints is None and thymio is not None:
                waypoints, samples = compute_path(obstacles, thymio, goal_mm, ARENA_W, ARENA_H)

            # 4. Display: inflated obstacles, waypoints and sampled points
            view = draw_obstacles(frame.copy(), obstacles, inflated_obstacles, to_camera)
            view = draw_path(view, waypoints, samples, to_camera)
            cv2.imshow("Global navigation", draw_legend(view))

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):                 # quit
                break
            if key == ord("p") and thymio is not None:   # replan (e.g. after a kidnapping)
                waypoints, samples = compute_path(obstacles, thymio, goal_mm, ARENA_W, ARENA_H)
            if key == ord("r"):                 # rerun the vision (pieces moved)
                try:
                    H, to_camera, obstacles, inflated_obstacles, goal_mm = init_obstacles(frame)
                    waypoints = samples = None
                except GridNotDetectedError as error:
                    print(error)

    except GridNotDetectedError as error:
        print(error)
        cv2.imshow("Grid not detected (press a key to quit)", frame)
        cv2.waitKey(0)
    finally:
        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()